# Student Performance Decision-Support System

## Tree Models Notebook

This notebook trains and evaluates decision tree and random forest
classifiers on both project experiments (early_warning and
progress_informed), using the shared train-test split and evaluation
functions established in the foundation notebook and `src/evaluation.py`.

**Owner:** Selorm Kwame Hlodze — Tree Models and Final Evaluation Lead


## 1. Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Find the repository root
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Allow imports from the src folder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("src folder exists:", (PROJECT_ROOT / "src").exists())

from src.config import (
    RANDOM_STATE,
    TARGET_COLUMN,
    TRAIN_INDICES_PATH,
    TEST_INDICES_PATH,
    EARLY_WARNING_EXPERIMENT,
    PROGRESS_INFORMED_EXPERIMENT,
)

from src.data import (
    load_student_data,
    create_target,
    get_feature_sets,
)

from src.preprocessing import build_model_pipeline

from src.evaluation import (
    evaluate_classifier,
    cross_validate_classifier,
    attach_cv_results,
    plot_confusion_matrix,
    compare_model_results,
    evaluate_subgroups,
)

print("Project imports completed successfully.")


## 2. Load the shared dataset, feature sets, and split

In [ ]:
# Rebuild the dataset and feature sets exactly as the foundation notebook did
df_raw = load_student_data()
df = create_target(df_raw)

X_base, X_early, X_progress, y = get_feature_sets(df)

print("Base feature shape:", X_base.shape)
print("Early-warning feature shape:", X_early.shape)
print("Progress-informed feature shape:", X_progress.shape)


In [ ]:
# Identify numeric and categorical columns the same way the foundation notebook did
numeric_features = (
    X_base
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

categorical_features = (
    X_base
    .select_dtypes(include=["object", "category", "bool"])
    .columns
    .tolist()
)

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


In [ ]:
# Load the shared row indices saved by the foundation notebook
train_indices = pd.read_csv(TRAIN_INDICES_PATH).iloc[:, 0].values
test_indices = pd.read_csv(TEST_INDICES_PATH).iloc[:, 0].values

print("Training rows:", len(train_indices))
print("Testing rows:", len(test_indices))


In [ ]:
def make_split(X):
    X_train = X.loc[train_indices]
    X_test = X.loc[test_indices]
    y_train = y.loc[train_indices]
    y_test = y.loc[test_indices]
    return X_train, X_test, y_train, y_test

# Early-warning split (no G1, G2)
X_train_early, X_test_early, y_train_early, y_test_early = make_split(X_early)

# Progress-informed split (includes G1, G2)
X_train_prog, X_test_prog, y_train_prog, y_test_prog = make_split(X_progress)

print("Early-warning train/test:", X_train_early.shape, X_test_early.shape)
print("Progress-informed train/test:", X_train_prog.shape, X_test_prog.shape)


In [ ]:
# Collect results from all four combinations here
all_results = []


## 3. Decision Tree — early_warning experiment

Baseline first, then tuned. Record why each hyperparameter is changed
rather than just grid-searching blindly.


In [ ]:
# Baseline decision tree, default hyperparameters
tree_pipeline_baseline = build_model_pipeline(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    classifier=DecisionTreeClassifier(random_state=RANDOM_STATE),
)

result_tree_early_baseline = evaluate_classifier(
    tree_pipeline_baseline,
    X_train_early, X_test_early, y_train_early, y_test_early,
    experiment_name=EARLY_WARNING_EXPERIMENT,
    model_name="decision_tree_baseline",
)

print("Training accuracy:", result_tree_early_baseline["training_accuracy"])
print("Testing accuracy:", result_tree_early_baseline["testing_accuracy"])
print("Recall:", result_tree_early_baseline["support_recall"])


**Overfitting check:** compare training vs. testing accuracy above.
A default, unconstrained decision tree usually overfits heavily
(training accuracy close to 1.0, testing accuracy much lower) — this is
expected and is the reason we tune `max_depth` / `min_samples_split` /
`min_samples_leaf` below rather than using the baseline as final.


In [ ]:
# Tuned decision tree
# max_depth limits how deep the tree can grow, directly controlling overfitting
# min_samples_leaf forces each leaf to represent a meaningful number of students
# TODO: try a couple of values here and justify the final choice in a markdown cell
tree_pipeline_early = build_model_pipeline(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    classifier=DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        max_depth=5,
        min_samples_leaf=10,
    ),
)

result_tree_early = evaluate_classifier(
    tree_pipeline_early,
    X_train_early, X_test_early, y_train_early, y_test_early,
    experiment_name=EARLY_WARNING_EXPERIMENT,
    model_name="decision_tree",
)

cv_summary_tree_early = cross_validate_classifier(
    tree_pipeline_early, X_train_early, y_train_early
)
result_tree_early = attach_cv_results(result_tree_early, cv_summary_tree_early)

print(result_tree_early)


In [ ]:
plot_confusion_matrix(
    result_tree_early["confusion_matrix"],
    title="Decision Tree — Early Warning",
)

all_results.append(result_tree_early)


## 4. Decision Tree — progress_informed experiment

In [ ]:
# Same tuning approach as the early-warning tree, applied to the progress-informed features
# Note: G1/G2 are strong predictors of G3, so this model is expected to score noticeably
# higher than the early-warning version — worth flagging in the summary, not treating as
# a fairer comparison.
tree_pipeline_prog = build_model_pipeline(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    classifier=DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        max_depth=5,
        min_samples_leaf=10,
    ),
)

result_tree_prog = evaluate_classifier(
    tree_pipeline_prog,
    X_train_prog, X_test_prog, y_train_prog, y_test_prog,
    experiment_name=PROGRESS_INFORMED_EXPERIMENT,
    model_name="decision_tree",
)

cv_summary_tree_prog = cross_validate_classifier(
    tree_pipeline_prog, X_train_prog, y_train_prog
)
result_tree_prog = attach_cv_results(result_tree_prog, cv_summary_tree_prog)

print(result_tree_prog)


In [ ]:
plot_confusion_matrix(
    result_tree_prog["confusion_matrix"],
    title="Decision Tree — Progress Informed",
)

all_results.append(result_tree_prog)


## 5. Random Forest — early_warning experiment

Random forests are less prone to overfitting than a single tree, but the
train/test gap should still be checked. Key hyperparameters here are
`n_estimators`, `max_depth`, and `class_weight` (useful if recall on the
minority class needs a boost).


In [ ]:
# n_estimators controls how many trees are averaged — more trees generally
# means more stable predictions, at the cost of training time
# class_weight="balanced" up-weights the minority class, which can help recall
forest_pipeline_early = build_model_pipeline(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    classifier=RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_estimators=200,
        max_depth=8,
        class_weight="balanced",
    ),
)

result_forest_early = evaluate_classifier(
    forest_pipeline_early,
    X_train_early, X_test_early, y_train_early, y_test_early,
    experiment_name=EARLY_WARNING_EXPERIMENT,
    model_name="random_forest",
)

cv_summary_forest_early = cross_validate_classifier(
    forest_pipeline_early, X_train_early, y_train_early
)
result_forest_early = attach_cv_results(result_forest_early, cv_summary_forest_early)

print(result_forest_early)


In [ ]:
plot_confusion_matrix(
    result_forest_early["confusion_matrix"],
    title="Random Forest — Early Warning",
)

all_results.append(result_forest_early)


## 6. Random Forest — progress_informed experiment

In [ ]:
forest_pipeline_prog = build_model_pipeline(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    classifier=RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_estimators=200,
        max_depth=8,
        class_weight="balanced",
    ),
)

result_forest_prog = evaluate_classifier(
    forest_pipeline_prog,
    X_train_prog, X_test_prog, y_train_prog, y_test_prog,
    experiment_name=PROGRESS_INFORMED_EXPERIMENT,
    model_name="random_forest",
)

cv_summary_forest_prog = cross_validate_classifier(
    forest_pipeline_prog, X_train_prog, y_train_prog
)
result_forest_prog = attach_cv_results(result_forest_prog, cv_summary_forest_prog)

print(result_forest_prog)


In [ ]:
plot_confusion_matrix(
    result_forest_prog["confusion_matrix"],
    title="Random Forest — Progress Informed",
)

all_results.append(result_forest_prog)


## 7. Combine all four results

In [ ]:
model_results_df = compare_model_results(all_results)
model_results_df


## 8. Fairness check (section 5.9)

In [ ]:
# Run this against whichever pipeline scores best above once that's decided.
# Example using the early-warning random forest pipeline:

sex_results = evaluate_subgroups(
    forest_pipeline_early, X_test_early, y_test_early,
    subgroup_column=df.loc[X_test_early.index, "sex"],
    group_name="sex",
)
sex_results


In [ ]:
school_results = evaluate_subgroups(
    forest_pipeline_early, X_test_early, y_test_early,
    subgroup_column=df.loc[X_test_early.index, "school"],
    group_name="school",
)
school_results


In [ ]:
address_results = evaluate_subgroups(
    forest_pipeline_early, X_test_early, y_test_early,
    subgroup_column=df.loc[X_test_early.index, "address"],
    group_name="address",
)
address_results


## 9. Section Summary

- Four model-experiment combinations trained and evaluated: decision tree
  and random forest, each on early_warning and progress_informed features.
- `model_results_df` above combines all four for comparison.
- TODO once run: note which combination has the strongest recall, whether
  any model shows overfitting (train vs. test gap), and whether
  progress_informed models score meaningfully higher due to G1/G2 (expected).
- TODO: confirm fairness check results don't show a concerning gap across
  sex / school / address before recommending a final model in
  `04_final_evaluation.ipynb`.
